# ClassCounter Examples

This notebook demonstrates how to use `classcounter` to count occurrences of class values in arrays and tensors.

In [1]:
import numpy as np

from classcounter import count_classes

## Basic Usage

Pass a numpy array to `count_classes` and get back a dict mapping each unique value to its count.

In [2]:
rng = np.random.default_rng(42)
arr = rng.integers(0, 5, size=(100, 100), dtype=np.int32)

count_classes(arr)

{0: 2022, 1: 1993, 2: 2077, 3: 1967, 4: 1941}

## Name Mapping

Use the `names` parameter to map integer class IDs to human-readable labels. Unmapped classes are preserved with their integer key (a warning is emitted).

In [3]:
names = {0: "water", 1: "forest", 2: "urban", 3: "crop", 4: "bare"}

count_classes(arr, names=names)

{'water': 2022, 'forest': 1993, 'urban': 2077, 'crop': 1967, 'bare': 1941}

## Percentage Output

Set `percent=True` to get results as percentages (0–100) instead of raw counts. This combines with `names` too.

In [4]:
count_classes(arr, names=names, percent=True)

{'water': 20.22, 'forest': 19.93, 'urban': 20.77, 'crop': 19.67, 'bare': 19.41}

## PyTorch Tensors

Pass a torch tensor and the torch backend is used automatically. Install with `pip install classcounter[torch]`.

In [5]:
import torch

tensor = torch.randint(0, 5, (100, 100))
count_classes(tensor)

{0: 1944, 1: 2004, 2: 2037, 3: 2004, 4: 2011}

## Performance

With `classcounter[numba]` installed, numpy arrays are automatically counted with a JIT-compiled loop — no code changes needed. PyTorch is another fast option, especially on GPU.

In [15]:
big = rng.integers(0, 10, size=(20000, 20000), dtype=np.int32)
print(f"Elements: {big.size:,}")

%timeit count_classes(big)

Elements: 400,000,000
263 ms ± 14.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

big_tensor = torch.randint(0, 10, (20000, 20000), device=device)

# CPU baseline
big_cpu = big_tensor.cpu()
%timeit count_classes(big_cpu)

# GPU (or same CPU if no GPU available)
%timeit count_classes(big_tensor)

Using device: cuda
802 ms ± 63.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
33.2 ms ± 600 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## GeoTIFF Support

Install with `pip install classcounter[geo]`. Pass a file path directly to count classes in a raster.

In [8]:
import tempfile
from pathlib import Path

import rasterio
from rasterio.transform import from_bounds

# Create a small land-cover raster (5 classes, 100x100 pixels)
land_cover = rng.integers(0, 5, size=(100, 100), dtype=np.uint8)
sample_tif = Path(tempfile.mkdtemp()) / "land_cover.tif"
transform = from_bounds(115.0, -33.0, 116.0, -32.0, 100, 100)

with rasterio.open(
    sample_tif,
    "w",
    driver="GTiff",
    height=100,
    width=100,
    count=1,
    dtype="uint8",
    transform=transform,
    crs="EPSG:4326",
) as dst:
    dst.write(land_cover, 1)

print(f"Created: {sample_tif}")

Created: /tmp/tmpr0hwn1go/land_cover.tif


In [9]:
count_classes(sample_tif, names=names)

{'water': 1932, 'forest': 2068, 'urban': 2024, 'crop': 1978, 'bare': 1998}

## Saving Results to GeoTIFF Metadata

Use `save_metadata=True` to write class counts back into the raster's GDAL metadata tags (one tag per class, e.g. `CLASS_COUNT_0=1234`). For more control, use the standalone `save_counts_to_raster` function.

In [10]:
from classcounter import save_counts_to_raster

# One-liner: count and write CLASS_COUNT_ tags in one step
count_classes(sample_tif, save_metadata=True)

{0: 1932, 1: 2068, 2: 2024, 3: 1978, 4: 1998}

In [11]:
# Verify the tags were written
with rasterio.open(sample_tif) as src:
    tags = src.tags()

{k: v for k, v in tags.items() if k.startswith("CLASS_")}

{'CLASS_COUNT_0': '1932',
 'CLASS_COUNT_1': '2068',
 'CLASS_COUNT_2': '2024',
 'CLASS_COUNT_3': '1978',
 'CLASS_COUNT_4': '1998'}

With `percent=True` the prefix switches to `CLASS_PERCENT_`. You can also set a custom prefix with `metadata_prefix`, or call `save_counts_to_raster` directly:

In [12]:
# Percent metadata
count_classes(sample_tif, percent=True, save_metadata=True)

# Custom prefix
count_classes(sample_tif, save_metadata=True, metadata_prefix="LAND_")

# Standalone function with named classes
counts = count_classes(sample_tif, names=names)
save_counts_to_raster(sample_tif, counts, prefix="NAMED_")